# 02 — Предобработка данных и Baseline-модели

**Бейзлайн:** простая модель логистической регрессии без генерации новых признаков (feature engineering).
Обучаем модель на всех финансовых показателях без сложного отбора.

## 0. Настройка

In [1]:
import sys
sys.path.insert(0, "..")
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocessing import load_raw_data, encode_target, drop_duplicates, handle_missing, clip_outliers, split_data, save_processed, RAW_FEATURES, TARGET_BIN, SEED
from src.modeling import train_logistic_baseline
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 30)

## 1. Загрузка и очистка

In [2]:
raw = load_raw_data("../data/raw/american_bankruptcy.csv")
df = encode_target(raw)
df = drop_duplicates(df)
df = handle_missing(df)
df = clip_outliers(df, RAW_FEATURES, lower_q=0.01, upper_q=0.99)
print(f"Итог: {df.shape[0]:,} строк, {df.shape[1]} столбцов")

Удалено дубликатов: 0
Нет пропусков
Итог: 78,682 строк, 22 столбцов


## 2. Группы: Train / Validation / Test Split

Разбиваем данные со стратификацией по целевой переменной.

In [3]:
X = df[RAW_FEATURES].copy()
y = df[TARGET_BIN].copy()
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, val_size=0.15, test_size=0.15, stratify=True, seed=SEED)
save_processed(X_train, X_val, X_test, y_train, y_val, y_test)

  Размеры выборок — train: 55076, val: 11803, test: 11803
  Доля rate — train: 0.066, val: 0.066, test: 0.066
  Сохранено в D:\hseml-group-project-yuliasx\data\processed


## 3. Baseline: Логистическая регрессия

Оцениваем базовую предсказательную способность. Используем `class_weight='balanced'`, так как без этого модель в условиях дисбаланса почти всегда будет предсказывать класс класс alive.

In [4]:
log_res = train_logistic_baseline(X_train, y_train, X_val, y_val, X_test, y_test)
print(f"Test ROC-AUC: {log_res['test_roc_auc']:.4f} | PR-AUC: {log_res['test_pr_auc']:.4f}")
print(log_res['test_report'])

Test ROC-AUC: 0.6811 | PR-AUC: 0.1369
              precision    recall  f1-score   support

       alive       0.97      0.37      0.54     11020
    bankrupt       0.09      0.84      0.16       783

    accuracy                           0.40     11803
   macro avg       0.53      0.60      0.35     11803
weighted avg       0.91      0.40      0.51     11803



In [6]:
if "coef" in log_res:
    print("Топ-10 признаков по абсолютному коэффициенту:")
    print(log_res["coef"].head(10).round(4).to_string())

Топ-10 признаков по абсолютному коэффициенту:
X8    -1.3432
X1    -0.8939
X14    0.6885
X12   -0.5380
X3     0.4355
X10    0.2892
X15   -0.2850
X6    -0.2792
X4     0.1775
X13    0.1485
